# OmniCare Clinical Copilot - Notebook 2: Admission & Vitals (Multi-Agent)

**Pipeline:** Scenario FHIR Bundle -> Google Cloud Healthcare API FHIR Store -> VitalsMonitorAgent -> Anomaly Detection -> Admission Note

**Architecture:** Uses `ClinicalOrchestrator` and `VitalsMonitorAgent` from the multi-agent framework.
All data persisted in Firestore with vitals/conditions/medications subcollections.

**Prerequisites:**
- Run `00_setup_and_models.ipynb` (models + orchestrator loaded)
- Run `01_consultation_audio_soap.ipynb` (encounter created, SOAP note generated)

## 0. Colab Bootstrap (run this first)

Auto-detects environment. In Colab, clones the private repo using your GitHub PAT.

**One-time setup:** In Colab, go to the **Key icon** in the left sidebar > Add a secret named `GITHUB_PAT` with your [GitHub Personal Access Token](https://github.com/settings/tokens).

In [ ]:
# ===========================================================
# Colab Bootstrap - run this cell first (works locally too)
# ===========================================================
import os, sys

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_DIR = '/content/omnicare-clinical-copilot'

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        token = userdata.get('GITHUB_PAT')
        repo_url = f'https://{token}@github.com/Yashground/omnicare-clinical-copilot.git'
        os.system(f'git clone {repo_url} {REPO_DIR}')
    NOTEBOOKS_DIR = os.path.join(REPO_DIR, 'notebooks')
    os.makedirs('/content/encounters', exist_ok=True)
    os.makedirs('/content/sample_data', exist_ok=True)
else:
    NOTEBOOKS_DIR = os.path.dirname(os.path.abspath('__file__'))

if NOTEBOOKS_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOKS_DIR)

print(f'Environment ready | Colab: {IN_COLAB} | Notebooks dir: {NOTEBOOKS_DIR}')

## 1. Load Encounter from Notebook 01 (Firestore)

Retrieve the encounter created in Notebook 01. The orchestrator from Notebook 00
is reused and pointed at the existing encounter.

In [ ]:
import os
import sys
import json
from datetime import datetime

from utils.encounter_state import load_encounter, list_encounters
from utils.firestore_db import load_encounter as fs_load_encounter, get_vitals, get_medications, get_conditions
from utils.patient_simulator import get_scenario, list_scenarios, generate_fhir_bundle
from utils.fhir_helpers import (
    parse_fhir_bundle, extract_vitals, extract_conditions,
    extract_medications, extract_patient_demographics, format_vitals_summary,
)
from utils.healthcare_api import (
    upload_fhir_bundle, query_vitals, query_conditions, query_medications, get_patient,
)
from utils.agents import ClinicalOrchestrator, VitalsMonitorAgent

# --- Verify orchestrator from Notebook 00 ---
try:
    _ = orchestrator
    _ = models
    print(f"Orchestrator ready with agents: {list(orchestrator.agents.keys())}")
except NameError:
    raise RuntimeError("orchestrator / models not found. Run Notebook 00 first!")

# --- Load the most recent encounter from Firestore ---
available = list_encounters()
if not available:
    raise RuntimeError("No encounters found. Run Notebook 01 first!")
encounter_id = available[-1]

encounter = load_encounter(encounter_id)
soap_note = encounter["stages"]["consultation"].get("soap_note", {})
patient_info = encounter.get("patient", {})

# Point the orchestrator at this encounter
orchestrator.encounter_id = encounter_id

# Determine which scenario matches this patient (if any)
scenario_id = None
for s in list_scenarios():
    if s["patient"] == patient_info.get("name"):
        scenario_id = s["id"]
        break

if scenario_id:
    scenario = get_scenario(scenario_id)
    orchestrator.scenario = scenario
    print(f"Matched scenario: {scenario_id} ({scenario['chief_complaint']})")
else:
    # Default to first scenario
    scenario = get_scenario("pneumonia_diabetic")
    orchestrator.scenario = scenario
    print(f"No exact match; using default scenario: {scenario['id']}")

# Inject consultation results so orchestrator has them for admission
orchestrator._results["consultation"] = {
    "transcript": encounter["stages"]["consultation"].get("transcript", ""),
    "soap_note": soap_note,
    "soap_raw": "",
}

print(f"\nEncounter: {encounter_id}")
print(f"Patient:   {patient_info.get('name', 'N/A')} (MRN: {patient_info.get('mrn', 'N/A')})")
print(f"SOAP note: {'present' if any(soap_note.values()) else 'empty'}")

## 2. Generate FHIR Bundle from Scenario

Dynamically build a FHIR R4 Bundle (Patient, Observations, Conditions, MedicationRequests)
from the matched patient scenario -- no hardcoded JSON.

In [ ]:
# Generate FHIR bundle dynamically from the scenario
fhir_bundle = generate_fhir_bundle(scenario)

# Summarise what was generated
resource_counts = {}
for entry in fhir_bundle.get("entry", []):
    rt = entry["resource"]["resourceType"]
    resource_counts[rt] = resource_counts.get(rt, 0) + 1

print(f"Generated FHIR Bundle ({fhir_bundle['type']}):")
for rt, count in resource_counts.items():
    print(f"  {rt}: {count}")
print(f"  Total entries: {len(fhir_bundle['entry'])}")

# Also save locally for reference
import tempfile
bundle_path = os.path.join(tempfile.gettempdir(), f"fhir_bundle_{encounter_id}.json")
with open(bundle_path, "w") as f:
    json.dump(fhir_bundle, f, indent=2)
print(f"\nBundle saved locally: {bundle_path}")

## 3. Upload to Google Cloud Healthcare API FHIR Store

Upload the generated bundle to the real FHIR store (created in Notebook 00).
The Healthcare API automatically converts the collection bundle to a transaction.

In [ ]:
# Upload bundle to Google Cloud Healthcare API FHIR store
print("Uploading FHIR bundle to Healthcare API...")
try:
    upload_result = upload_fhir_bundle(fhir_bundle)
    upload_entries = upload_result.get("entry", [])
    print(f"Upload successful: {len(upload_entries)} resources created/updated")

    # Show status of each uploaded resource
    for entry in upload_entries[:5]:
        resp = entry.get("response", {})
        print(f"  {resp.get('status', '?')} - {resp.get('location', 'N/A')}")
    if len(upload_entries) > 5:
        print(f"  ... and {len(upload_entries) - 5} more")
except Exception as e:
    print(f"FHIR store upload failed: {e}")
    print("Continuing with local bundle parsing for downstream steps.")

## 4. Query Patient Data from FHIR Store

Read the data back from the Healthcare API FHIR store to verify the upload
and to show the round-trip works. Falls back to local bundle parsing if
the FHIR store is unreachable.

In [ ]:
patient_id = scenario["demographics"]["mrn"]
fhir_query_ok = False

try:
    print(f"Querying FHIR store for patient: {patient_id}\n")

    # Query vitals
    raw_vitals = query_vitals(patient_id)
    fhir_vitals = extract_vitals(raw_vitals)
    print(f"Vitals from FHIR store: {len(fhir_vitals)} readings")
    for v in fhir_vitals:
        print(f"  - {v['display']}: {v['value']} {v['unit']}")

    # Query conditions
    raw_conditions = query_conditions(patient_id)
    fhir_conditions = extract_conditions(raw_conditions)
    print(f"\nConditions from FHIR store: {len(fhir_conditions)}")
    for c in fhir_conditions:
        print(f"  - {c['display']} (onset: {c['onset']}, status: {c['status']})")

    # Query medications
    raw_medications = query_medications(patient_id)
    fhir_medications = extract_medications(raw_medications)
    print(f"\nMedications from FHIR store: {len(fhir_medications)}")
    for m in fhir_medications:
        print(f"  - {m['display']} ({m['status']})")

    # Query patient demographics
    fhir_patient = get_patient(patient_id)
    if fhir_patient:
        demographics = extract_patient_demographics(fhir_patient)
        print(f"\nPatient: {demographics.get('name', 'N/A')}, "
              f"DOB: {demographics.get('dob', 'N/A')}, "
              f"Gender: {demographics.get('gender', 'N/A')}")

    fhir_query_ok = True
    print("\nFHIR store round-trip verified successfully.")

except Exception as e:
    print(f"FHIR store query failed: {e}")
    print("Falling back to local bundle parsing...")

    # Parse from the locally saved bundle file
    resources = parse_fhir_bundle(bundle_path)
    demographics = (extract_patient_demographics(resources["patients"][0])
                    if resources["patients"] else {})
    fhir_vitals = extract_vitals(resources["observations"])
    fhir_conditions = extract_conditions(resources["conditions"])
    fhir_medications = extract_medications(resources["medications"])
    print(f"\nLocal parse: {len(fhir_vitals)} vitals, "
          f"{len(fhir_conditions)} conditions, "
          f"{len(fhir_medications)} medications")

## 5. Run Vitals Monitor Agent

Use the orchestrator's `run_admission` method which delegates to the
`VitalsMonitorAgent`. When `use_fhir_server=True`, the agent queries vitals,
conditions, and medications directly from the Healthcare API FHIR store.

The agent:
1. Fetches clinical data from FHIR store (or local bundle as fallback)
2. Detects vital sign anomalies against normal ranges
3. Generates an admission note via MedGemma
4. Persists everything to Firestore (including vitals subcollection)

In [ ]:
# Run the admission phase via the orchestrator
# use_fhir_server=True tells VitalsMonitorAgent to query the Healthcare API
# Falls back to scenario-based bundle generation if FHIR store is unavailable
admission_result = orchestrator.run_admission(
    fhir_bundle_path=bundle_path,
    use_fhir_server=fhir_query_ok,
    patient_id=patient_id,
)

print(f"\nAgent returned:")
print(f"  Vitals:      {len(admission_result['vitals'])} readings")
print(f"  Conditions:  {len(admission_result['conditions'])}")
print(f"  Medications: {len(admission_result['medications'])}")
print(f"  Anomalies:   {len(admission_result['anomalies'])}")
print(f"  Note length: {len(admission_result['admission_note'])} chars")

## 6. Display Anomalies & Vitals Summary

Show the VitalsMonitorAgent's anomaly detection results alongside the
full vitals, conditions, and medications extracted from the FHIR store.

In [ ]:
# --- Anomaly Detection Results ---
anomalies = admission_result["anomalies"]
if anomalies:
    print("VITAL SIGN ANOMALIES DETECTED:")
    print("-" * 50)
    for a in anomalies:
        status_marker = "[HIGH]" if a["status"] == "HIGH" else "[LOW] "
        print(f"  {status_marker} {a['message']}")
    print()
else:
    print("No vital sign anomalies detected.\n")

# --- Vitals Summary ---
print("Current Vital Signs:")
print("-" * 50)
vitals_summary = admission_result.get("vitals_summary", format_vitals_summary(admission_result["vitals"]))
print(vitals_summary)

# --- Conditions ---
print("\nActive Conditions:")
print("-" * 50)
for c in admission_result["conditions"]:
    print(f"  - {c['display']} (onset: {c.get('onset', 'N/A')}, status: {c.get('status', 'N/A')})")

# --- Medications ---
print("\nCurrent Medications:")
print("-" * 50)
for m in admission_result["medications"]:
    print(f"  - {m['display']} ({m.get('status', 'N/A')})")

# --- Normal ranges reference ---
print("\nReference Ranges (VitalsMonitorAgent.NORMAL_RANGES):")
print("-" * 50)
for name, (low, high, unit) in VitalsMonitorAgent.NORMAL_RANGES.items():
    print(f"  {name}: {low}-{high} {unit}")

## 7. Display Admission Note

The admission note was generated by VitalsMonitorAgent using MedGemma,
incorporating the consultation SOAP note, vitals, conditions, and medications.

In [ ]:
admission_note = admission_result["admission_note"]

print("=" * 60)
print("ADMISSION NOTE (Generated by VitalsMonitorAgent + MedGemma)")
print("=" * 60)
print(admission_note)
print("=" * 60)
print(f"\nNote length: {len(admission_note)} characters")

## 8. View Encounter State (Firestore)

The VitalsMonitorAgent already persisted everything to Firestore:
- Admission stage data (vitals, conditions, medications, admission note, anomalies)
- Vitals subcollection (individual vital readings, queryable)
- Medications subcollection
- Conditions subcollection
- Agent activity logs

Verify the data was stored correctly.

In [ ]:
# --- Reload encounter from Firestore to verify persistence ---
enc = fs_load_encounter(encounter_id)
admission_stage = enc["stages"]["admission"]

print(f"Encounter: {encounter_id}")
print(f"Status:    {enc.get('status', 'N/A')}")
print(f"Updated:   {enc.get('updated_at', 'N/A')}")

print(f"\nAdmission stage (Firestore):")
print(f"  FHIR Patient ID:  {admission_stage.get('fhir_patient_id', 'N/A')}")
print(f"  Vitals stored:    {len(admission_stage.get('vitals_history', []))}")
print(f"  Conditions:       {len(admission_stage.get('conditions', []))}")
print(f"  Medications:      {len(admission_stage.get('medications', []))}")
print(f"  Anomalies:        {len(admission_stage.get('anomalies', []))}")
print(f"  Admission note:   {'present' if admission_stage.get('admission_note') else 'missing'}")
print(f"  Timestamp:        {admission_stage.get('timestamp', 'N/A')}")

# --- Query subcollections ---
print(f"\nFirestore subcollections for {encounter_id}:")

fs_vitals = get_vitals(encounter_id)
print(f"  vitals:      {len(fs_vitals)} documents")
for v in fs_vitals[:3]:
    print(f"    - {v['display']}: {v['value']} {v['unit']} @ {v.get('recorded_at', 'N/A')}")
if len(fs_vitals) > 3:
    print(f"    ... and {len(fs_vitals) - 3} more")

fs_meds = get_medications(encounter_id)
print(f"  medications: {len(fs_meds)} documents")
for m in fs_meds:
    print(f"    - {m['display']} ({m.get('status', 'N/A')})")

fs_conds = get_conditions(encounter_id)
print(f"  conditions:  {len(fs_conds)} documents")
for c in fs_conds:
    print(f"    - {c['display']} ({c.get('status', 'N/A')})")

# --- Agent activity logs ---
from utils.firestore_db import get_agent_logs
logs = get_agent_logs(encounter_id, agent_name="vitals_monitor_agent")
print(f"\nVitalsMonitorAgent logs: {len(logs)} entries")
for log in logs:
    print(f"  [{log['timestamp']}] {log['action']}: {log.get('details', '')}")

# --- Next step ---
print(f"\nAdmission stage complete!")
print(f"Next: Run 03_radiology_dicom_imaging.ipynb with encounter_id = '{encounter_id}'")